In [10]:
#nullable enable

public interface IInstallable
{
    void Install(Computer computer);
}

public abstract class Component
{
    public string Name {get; set;}
    public decimal Price {get; set;}

    public Component(string name, decimal price)
    {
        Name = name;
        Price = price;
    }

    public virtual string GetInfo()
    {
        return $"{Name}, цена: {Price} руб.";
    }

    public override string ToString()
    {
        return GetInfo();
    }

    protected bool BaseEquals(Component? other)
    {
        return other != null && Name == other.Name && Price == other.Price;
    }

    protected int BaseHashCode()
    {
        return HashCode.Combine(Name, Price);
    }
}

public class CPU : Component, IInstallable
{
    public int Cores {get; set;}
    public double Frequency {get; set;}

    public CPU(string name, decimal price, int cores, double frequency) : base(name, price)
    {
        Cores = cores;
        Frequency = frequency;
    }

    public override string GetInfo()
    {
        return $"CPU: {Name}, ядер: {Cores}, частота: {Frequency} ГГц, цена: {Price} руб.";
    }

    public void Install(Computer computer)
    {
        Console.WriteLine($"Установка процессора: {Name}");
    }

    public static bool operator ==(CPU? a, CPU? b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;

        return a.BaseEquals(b) && a.Cores == b.Cores && a.Frequency == b.Frequency;
    }

    public static bool operator !=(CPU? a, CPU? b) => !(a == b);

    public override bool Equals(object? obj)
    {
        return obj is CPU cpu && this == cpu;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(BaseHashCode(), Cores, Frequency);
    }
}

public class RAM : Component, IInstallable
{
    public int Capacity { get; set; }
    public int Frequency { get; set; }

    public RAM(string name, decimal price, int capacity, int frequency) : base(name, price)
    {
        Capacity = capacity;
        Frequency = frequency;
    }

    public override string GetInfo()
    {
        return $"RAM: {Name}, объём: {Capacity} ГБ, частота: {Frequency} МГц, цена: {Price} руб.";
    }

    public void Install(Computer computer)
    {
        Console.WriteLine($"Установка оперативной памяти: {Name}");
    }

    public static bool operator ==(RAM? a, RAM? b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;

        return a.BaseEquals(b) && a.Capacity == b.Capacity && a.Frequency == b.Frequency;
    }

    public static bool operator !=(RAM? a, RAM? b) => !(a == b);

    public override bool Equals(object? obj)
    {
        return obj is RAM ram && this == ram;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(BaseHashCode(), Capacity, Frequency);
    }
}

public class GPU : Component, IInstallable
{
    public int MemorySize { get; set; }

    public GPU(string name, decimal price, int memorySize) : base(name, price)
    {
        MemorySize = memorySize;
    }

    public override string GetInfo()
    {
        return $"GPU: {Name}, видеопамять: {MemorySize} ГБ, цена: {Price} руб.";
    }

    public void Install(Computer computer)
    {
        Console.WriteLine($"Установка видеокарты: {Name}");
    }

    public static bool operator ==(GPU? a, GPU? b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;

        return a.BaseEquals(b) && a.MemorySize == b.MemorySize;
    }

    public static bool operator !=(GPU? a, GPU? b) => !(a == b);

    public override bool Equals(object? obj)
    {
        return obj is GPU gpu && this == gpu;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(BaseHashCode(), MemorySize);
    }
}

public class HDD : Component, IInstallable
{
    public int Capacity { get; set; }
    public int RPM { get; set; }

    public HDD(string name, decimal price, int capacity, int rpm) : base(name, price)
    {
        Capacity = capacity;
        RPM = rpm;
    }

    public override string GetInfo()
    {
        return $"HDD: {Name}, объём: {Capacity} ГБ, скорость: {RPM} об/мин, цена: {Price} руб.";
    }

    public void Install(Computer computer)
    {
        Console.WriteLine($"Установка жёсткого диска: {Name}");
    }

    public static bool operator ==(HDD? a, HDD? b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;

        return a.BaseEquals(b) && a.Capacity == b.Capacity && a.RPM == b.RPM;
    }

    public static bool operator !=(HDD? a, HDD? b) => !(a == b);

    public override bool Equals(object? obj)
    {
        return obj is HDD hdd && this == hdd;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(BaseHashCode(), Capacity, RPM);
    }
}

public class Computer
{
    public List<Component> Components {get;} = new();

    public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        if (component == null) throw new ArgumentException(nameof(component));

        component.Install(this);
        Components.Add(component);
    }

    public void RemoveComponent<T>(T component) where T : Component
    {
        if (component == null) throw new ArgumentNullException(nameof(component));
        if (Components.Remove(component)) Console.WriteLine($"Компонент удалён: {component.Name}");
        else Console.WriteLine($"Компонент не найден: {component.Name}");
    }

    public decimal GetTotalPrice()
    {
        return Components.Sum(component => component.Price);
    }

    public void PrintInfo()
    {
        Console.WriteLine("Информация о компьютере:");
        foreach (Component component in Components)
            Console.WriteLine(component.GetInfo());
    }

    public void PrintInfo(string title)
    {
        Console.WriteLine(title);
        PrintInfo();
    }

    internal void AddComponentFromFactory(Component component)
    {
        if (component is not IInstallable installable) throw new ArgumentException("Компонент должен реализовывать IInstallable.");

        installable.Install(this);
        Components.Add(component);
    }
}

public static class ComputerFactory
{
    public static T CreateComputer<T>(params Component[] components) where T : Computer, new()
    {
        T computer = new T();

        foreach (Component component in components) computer.AddComponentFromFactory(component);

        return computer;
    }

    public static Computer CreateOfficeComputer()
    {
        return CreateComputer<Computer>(
            new CPU("Intel Core i5", 20000m, 6, 3.2),
            new RAM("Kingston", 5000m, 16, 3200),
            new HDD("Seagate", 6000m, 1000, 7200));
    }

    public static Computer CreateGamingComputer()
    {
        return CreateComputer<Computer>(
            new CPU("Intel Core i7", 35000m, 8, 4.0),
            new RAM("Corsair", 10000m, 32, 3600),
            new GPU("NVIDIA GeForce RTX", 60000m, 12),
            new HDD("WD Black", 8000m, 2000, 7200));
    }
}





Console.WriteLine("=== ОФИСНЫЙ КОМПЬЮТЕР ===");

Computer office = ComputerFactory.CreateOfficeComputer();

office.PrintInfo();
Console.WriteLine($"Общая стоимость: {office.GetTotalPrice()} руб.");

Console.WriteLine("\n=== ИГРОВОЙ КОМПЬЮТЕР ===");

Computer gaming = ComputerFactory.CreateGamingComputer();

gaming.PrintInfo("Компоненты игрового компьютера:");
Console.WriteLine($"Общая стоимость: {gaming.GetTotalPrice()} руб.");

Console.WriteLine("\n=== ДОБАВЛЕНИЕ КОМПОНЕНТА ===");

GPU newGPU = new GPU("AMD Radeon RX", 40000m, 16);
office.AddComponent(newGPU);

office.PrintInfo();
Console.WriteLine($"Стоимость после добавления: {office.GetTotalPrice()} руб.");

Console.WriteLine("\n=== УДАЛЕНИЕ КОМПОНЕНТА ===");

office.RemoveComponent(newGPU);

Console.WriteLine($"Стоимость после удаления: {office.GetTotalPrice()} руб.");

Console.WriteLine("\n=== СРАВНЕНИЕ КОМПОНЕНТОВ ===");

CPU cpu1 = new CPU("Intel Core i7", 30000m, 8, 3.6);
CPU cpu2 = new CPU("Intel Core i7", 30000m, 8, 3.6);
CPU cpu3 = new CPU("AMD Ryzen 7", 32000m, 8, 3.8);

Console.WriteLine($"cpu1 == cpu2: {cpu1 == cpu2}");
Console.WriteLine($"cpu1 != cpu3: {cpu1 != cpu3}");

Console.WriteLine("\n=== ToString() ===");
Console.WriteLine(cpu1);

Console.WriteLine("\n=== ПОЛИМОРФИЗМ ===");

List<Component> components = new()
{
    cpu1,
    new RAM("Kingston Fury", 7000m, 16, 3200),
    new GPU("NVIDIA RTX", 50000m, 12),
    new HDD("Seagate Barracuda", 5000m, 1000, 7200)
};

foreach (Component component in components)
    Console.WriteLine(component.GetInfo());


=== ОФИСНЫЙ КОМПЬЮТЕР ===
Установка процессора: Intel Core i5
Установка оперативной памяти: Kingston
Установка жёсткого диска: Seagate
Информация о компьютере:
CPU: Intel Core i5, ядер: 6, частота: 3.2 ГГц, цена: 20000 руб.
RAM: Kingston, объём: 16 ГБ, частота: 3200 МГц, цена: 5000 руб.
HDD: Seagate, объём: 1000 ГБ, скорость: 7200 об/мин, цена: 6000 руб.
Общая стоимость: 31000 руб.

=== ИГРОВОЙ КОМПЬЮТЕР ===
Установка процессора: Intel Core i7
Установка оперативной памяти: Corsair
Установка видеокарты: NVIDIA GeForce RTX
Установка жёсткого диска: WD Black
Компоненты игрового компьютера:
Информация о компьютере:
CPU: Intel Core i7, ядер: 8, частота: 4 ГГц, цена: 35000 руб.
RAM: Corsair, объём: 32 ГБ, частота: 3600 МГц, цена: 10000 руб.
GPU: NVIDIA GeForce RTX, видеопамять: 12 ГБ, цена: 60000 руб.
HDD: WD Black, объём: 2000 ГБ, скорость: 7200 об/мин, цена: 8000 руб.
Общая стоимость: 113000 руб.

=== ДОБАВЛЕНИЕ КОМПОНЕНТА ===
Установка видеокарты: AMD Radeon RX
Информация о компьютере:
CP